<a href="https://colab.research.google.com/github/Edubel77/enrutaia/blob/main/notebooks/D9_lab_2_4_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Lab 2.4 - Paso 1: pipeline de sentimiento preentrenado sobre los tweets del Día 8
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import f1_score, accuracy_score, classification_report

dispositivo = 0 if torch.cuda.is_available() else -1
print("GPU disponible:", torch.cuda.is_available())

# Leer los Parquet del subconjunto en español directamente por su ruta
RUTA = "hf://datasets/cardiffnlp/tweet_sentiment_multilingual@refs%2Fconvert%2Fparquet/spanish"
ds = load_dataset("parquet", data_files={
    "train":      f"{RUTA}/train/*.parquet",
    "validation": f"{RUTA}/validation/*.parquet",
    "test":       f"{RUTA}/test/*.parquet",
})
print({s: ds[s].num_rows for s in ds})   # deben ser 1839 / 324 / 870, como el Día 8
val = ds["validation"].to_pandas()
nombres = ["negative", "neutral", "positive"]             # 0, 1, 2 (confirmado el Día 8)

clasificador = pipeline("text-classification",
                        model="pysentimiento/robertuito-sentiment-analysis",
                        device=dispositivo)

print(clasificador(["qué gran día, me encanta", "no está nada mal", "este servicio es pésimo"]))

# Traducir las etiquetas del modelo (NEG/NEU/POS) a las del dataset (0/1/2)
traduccion = {"NEG": 0, "NEU": 1, "POS": 2}
salidas = clasificador(val["text"].tolist(), batch_size=32, truncation=True)
pred = np.array([traduccion[s["label"]] for s in salidas])

print(f"\nRoBERTuito (sin entrenar con estos datos) | macro-F1: {f1_score(val['label'], pred, average='macro'):.4f} | "
      f"exactitud: {accuracy_score(val['label'], pred):.4f}")
print("Referencias del Día 8 -> TF-IDF: 0.5978 | BiLSTM: 0.5083")
print(classification_report(val["label"], pred, target_names=nombres, digits=3))

GPU disponible: True


0000.parquet:   0%|          | 0.00/120k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/58.0k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

... (6 outputs left out)

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

[{'label': 'POS', 'score': 0.9805317521095276}, {'label': 'POS', 'score': 0.7641283869743347}, {'label': 'NEG', 'score': 0.9743136763572693}]

RoBERTuito (sin entrenar con estos datos) | macro-F1: 0.7766 | exactitud: 0.7778
Referencias del Día 8 -> TF-IDF: 0.5978 | BiLSTM: 0.5083
              precision    recall  f1-score   support

    negative      0.804     0.833     0.818       108
     neutral      0.716     0.676     0.695       108
    positive      0.809     0.824     0.817       108

    accuracy                          0.778       324
   macro avg      0.776     0.778     0.777       324
weighted avg      0.776     0.778     0.777       324



In [6]:
# Lab 2.4 - Paso 2: AutoTokenizer + AutoModel (BETO) por dentro
from transformers import AutoTokenizer, AutoModel

nombre = "dccuchile/bert-base-spanish-wwm-uncased"
tok = AutoTokenizer.from_pretrained(nombre)
beto = AutoModel.from_pretrained(nombre).eval()      # .eval(): modo uso, sin dropout

entrada = tok(["pon una alarma a las siete"], return_tensors="pt")
print("input_ids:     ", entrada["input_ids"][0].tolist())
print("tokens:        ", tok.convert_ids_to_tokens(entrada["input_ids"][0]))
print("attention_mask:", entrada["attention_mask"][0].tolist())

with torch.no_grad():                                 # sin calcular gradientes: solo predecir
    salida = beto(**entrada)
print("\nForma de la salida:", tuple(salida.last_hidden_state.shape), "-> (frases, tokens, valores por token)")

# ¿La misma palabra tiene vectores distintos según el contexto?
def vector_de(frase, palabra):
    enc = tok(frase, return_tensors="pt")
    with torch.no_grad():
        h = beto(**enc).last_hidden_state[0]
    tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
    return h[tokens.index(palabra)]

coseno = torch.nn.functional.cosine_similarity
banco_asiento  = vector_de("me senté en el banco del parque", "banco")
banco_asiento2 = vector_de("el banco de madera estaba mojado", "banco")
banco_dinero   = vector_de("el banco me negó el crédito", "banco")
print(f"\nbanco (parque) vs banco (madera):  {coseno(banco_asiento, banco_asiento2, dim=0):.3f}")
print(f"banco (parque) vs banco (crédito): {coseno(banco_asiento, banco_dinero, dim=0):.3f}")

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

... (3 outputs left out)

input_ids:      [4, 1732, 1091, 11420, 1012, 1085, 4018, 5]
tokens:         ['[CLS]', 'pon', 'una', 'alarma', 'a', 'las', 'siete', '[SEP]']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1]


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            


Forma de la salida: (1, 8, 768) -> (frases, tokens, valores por token)

banco (parque) vs banco (madera):  0.680
banco (parque) vs banco (crédito): 0.478


In [7]:
!pip install -q sentencepiece protobuf    # el tokenizador de mDeBERTa los necesita

In [8]:
# Lab 2.4 - Paso 3: E2 - zero-shot sobre la validación de EnrutaIA
import hashlib, io, json, os, time, urllib.request
from transformers import pipeline

BASE = "https://raw.githubusercontent.com/Edubel77/enrutaia/main/data/limpio/"
contenido = urllib.request.urlopen(BASE + "validation.csv").read()
assert hashlib.md5(contenido).hexdigest() == "ba68845308293612e717aa7c1a55887c", "¡validation.csv cambió!"
va = pd.read_csv(io.BytesIO(contenido))

# Una descripción en español para cada dominio (esto es lo que "lee" el modelo)
DESCRIPCIONES = {
    "alarm": "alarmas y despertador",
    "audio": "volumen y sonido del dispositivo",
    "calendar": "calendario, citas y recordatorios",
    "cooking": "recetas y cocina",
    "datetime": "la fecha y la hora",
    "email": "correo electrónico",
    "general": "conversación general, chistes y saludos",
    "iot": "dispositivos del hogar como luces, enchufes, aspiradora o cafetera",
    "lists": "listas de compras y de tareas",
    "music": "gustos e información sobre música",
    "news": "noticias",
    "play": "reproducir música, radio, podcasts o audiolibros",
    "qa": "preguntas de conocimiento general, definiciones, precios y datos",
    "recommendation": "recomendaciones de lugares, eventos o películas",
    "social": "redes sociales y quejas a empresas",
    "takeaway": "pedidos de comida a domicilio",
    "transport": "transporte, taxis, trenes, vuelos y tráfico",
    "weather": "el clima y el pronóstico del tiempo",
}
PLANTILLA = "Esta solicitud trata sobre {}."
a_dominio = {desc: dom for dom, desc in DESCRIPCIONES.items()}

zs = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
              device=0 if torch.cuda.is_available() else -1)

print(zs("pon una alarma a las siete", list(DESCRIPCIONES.values()),
         hypothesis_template=PLANTILLA)["labels"][:3])

inicio = time.time()
salidas = zs(va["text"].tolist(), list(DESCRIPCIONES.values()),
             hypothesis_template=PLANTILLA, batch_size=32)
seg = time.time() - inicio
pred_e2 = [a_dominio[s["labels"][0]] for s in salidas]

dominios = sorted(DESCRIPCIONES)
macro = f1_score(va["dominio"], pred_e2, average="macro")
f1_dom = f1_score(va["dominio"], pred_e2, average=None, labels=dominios)
print(f"\nE2 zero-shot | macro-F1: {macro:.4f} | exactitud: {accuracy_score(va['dominio'], pred_e2):.4f} | {seg:.0f} s")
print("E0 (TF-IDF):  macro-F1: 0.8725")
print("\nF1 por dominio (de peor a mejor):")
print(pd.Series(f1_dom, index=dominios).sort_values().round(3).to_string())

os.makedirs("results", exist_ok=True)
with open("results/E2_zeroshot.json", "w") as f:
    json.dump({"experimento": "E2", "modelo": "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
               "plantilla": PLANTILLA, "descripciones": DESCRIPCIONES,
               "split_evaluado": "validation", "macro_f1": round(macro, 4),
               "accuracy": round(accuracy_score(va["dominio"], pred_e2), 4),
               "f1_por_dominio": dict(zip(dominios, np.round(f1_dom, 4).tolist()))},
              f, indent=2, ensure_ascii=False)

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

['la fecha y la hora', 'volumen y sonido del dispositivo', 'alarmas y despertador']

E2 zero-shot | macro-F1: 0.2187 | exactitud: 0.2533 | 63 s
E0 (TF-IDF):  macro-F1: 0.8725

F1 por dominio (de peor a mejor):
general           0.000
iot               0.000
play              0.079
social            0.087
qa                0.089
lists             0.101
transport         0.130
alarm             0.145
audio             0.188
recommendation    0.211
takeaway          0.245
datetime          0.261
music             0.286
cooking           0.300
calendar          0.303
weather           0.390
news              0.457
email             0.665


In [10]:
# Lab 2.4 - Paso 3b: diagnóstico del zero-shot
# 1) ¿Predice casi siempre lo mismo?
print("Dominios más predichos por E2:")
print(pd.Series(pred_e2).value_counts().head(6).to_string())

# 2) ¿Cómo interpreta el pipeline las etiquetas del modelo?
print("\nEtiquetas del modelo NLI:", zs.model.config.id2label)

# 3) El ejemplo EXACTO de la ficha técnica del modelo: debería dar 'politics' primero
r = zs("Angela Merkel ist eine Politikerin in Deutschland und Vorsitzende der CDU",
       ["politics", "economy", "entertainment", "environment"])
print("\nEjemplo de la ficha técnica:", list(zip(r["labels"], [round(s, 3) for s in r["scores"]])))

# 4) Frases muy claras en español
for t in ["enciende las luces de la sala", "cómo estará el clima mañana",
          "pon música de shakira", "cuéntame un chiste"]:
    r = zs(t, list(DESCRIPCIONES.values()), hypothesis_template=PLANTILLA)
    print(f"{t:<32} -> {a_dominio[r['labels'][0]]:<10} ({r['scores'][0]:.2f})")

Dominios más predichos por E2:
transport    602
datetime     292
email        226
cooking      211
news         208
audio        134

Etiquetas del modelo NLI: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}

Ejemplo de la ficha técnica: [('politics', 0.966), ('economy', 0.023), ('environment', 0.007), ('entertainment', 0.004)]
enciende las luces de la sala    -> audio      (0.43)


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


cómo estará el clima mañana      -> weather    (0.63)
pon música de shakira            -> transport  (0.36)
cuéntame un chiste               -> transport  (0.35)


In [11]:
# Lab 2.4 - Paso 4: E2 con etiquetas cortas (misma plantilla, mismo modelo)
CORTAS = {
    "alarm": "alarmas", "audio": "volumen", "calendar": "calendario", "cooking": "cocina",
    "datetime": "fecha y hora", "email": "correo", "general": "conversación", "iot": "hogar inteligente",
    "lists": "listas", "music": "gustos musicales", "news": "noticias", "play": "reproducir contenido",
    "qa": "preguntas de cultura general", "recommendation": "recomendaciones", "social": "redes sociales",
    "takeaway": "pedir comida", "transport": "transporte", "weather": "clima",
}
a_dom_cortas = {v: k for k, v in CORTAS.items()}

salidas_c = zs(va["text"].tolist(), list(CORTAS.values()), hypothesis_template=PLANTILLA, batch_size=32)
pred_c = [a_dom_cortas[s["labels"][0]] for s in salidas_c]
macro_c = f1_score(va["dominio"], pred_c, average="macro")

print(f"E2 descripciones largas | macro-F1: {macro:.4f}")
print(f"E2 etiquetas cortas     | macro-F1: {macro_c:.4f} | diferencia: {macro_c - macro:+.4f}")
print("\nDominios más predichos con etiquetas cortas:")
print(pd.Series(pred_c).value_counts().head(5).to_string())
f1_c = f1_score(va["dominio"], pred_c, average=None, labels=dominios)
print("\nF1 de general:", round(f1_c[dominios.index("general")], 3), "| F1 de qa:", round(f1_c[dominios.index("qa")], 3))

with open("results/E2_zeroshot.json") as f:
    e2 = json.load(f)
e2["variante_etiquetas_cortas"] = {"etiquetas": CORTAS, "macro_f1": round(macro_c, 4)}
with open("results/E2_zeroshot.json", "w") as f:
    json.dump(e2, f, indent=2, ensure_ascii=False)

E2 descripciones largas | macro-F1: 0.2187
E2 etiquetas cortas     | macro-F1: 0.4357 | diferencia: +0.2170

Dominios más predichos con etiquetas cortas:
recommendation    240
social            206
calendar          162
audio             141
news              141

F1 de general: 0.099 | F1 de qa: 0.037
